In [42]:
import pandas as pd
from pathlib import Path

In [43]:
DATA_DIR = Path(
    r"C:\VAEP-Track\scripts\data\sportec_idsse"
)

MATCH_IDS = [
    "J03WPY",
    "J03WMX",
    "J03WN1",
    "J03WOH",
    "J03WOY",
    "J03WQQ",
    "J03WR9",
]

print("Data directory:", DATA_DIR)
print("Jumlah match:", len(MATCH_IDS))

Data directory: C:\VAEP-Track\scripts\data\sportec_idsse
Jumlah match: 7


In [44]:
def analyze_match(match_id):
    """
    Menganalisis satu pertandingan:
    - jumlah frame tracking
    - jumlah frame dengan ball_owning_team_id
    - jumlah perubahan possession
    - jumlah candidate saat bola aktif
    """

    tracking_path = DATA_DIR / f"{match_id}_tracking.csv"

    if not tracking_path.exists():
        print(f"[WARNING] File tidak ditemukan: {tracking_path}")
        return None

    # Load data
    tracking = pd.read_csv(tracking_path)

    # Timestamp
    tracking["timestamp"] = pd.to_timedelta(
        tracking["timestamp"],
        errors="coerce"
    )

    # Urutkan secara temporal
    tracking = tracking.sort_values(
        ["period_id", "timestamp", "frame_id"]
    ).reset_index(drop=True)

    # Tim penguasa bola pada frame sebelumnya
    tracking["prev_owning_team"] = (
        tracking
        .groupby("period_id")["ball_owning_team_id"]
        .shift(1)
    )

    # Perubahan possession
    tracking["team_changed"] = (
        tracking["ball_owning_team_id"].notna()
        & tracking["prev_owning_team"].notna()
        & (
            tracking["ball_owning_team_id"]
            != tracking["prev_owning_team"]
        )
    )

    # Semua perubahan possession
    possession_changes = tracking[
        tracking["team_changed"]
    ].copy()

    # Hanya ketika bola aktif
    active_changes = possession_changes[
        possession_changes["ball_state"] == "alive"
    ].copy()

    result = {
        "match_id": match_id,
        "tracking_frames": len(tracking),
        "owning_team_valid_frames":
            tracking["ball_owning_team_id"].notna().sum(),
        "all_possession_changes":
            len(possession_changes),
        "active_possession_changes":
            len(active_changes),
    }

    return result

In [45]:
print("=== HASIL DEBUG ===")

print("\nIsi results:")
print(results)

print("\nKolom summary_df:")
print(summary_df.columns.tolist())

print("\nIsi summary_df:")
display(summary_df)

=== HASIL DEBUG ===

Isi results:
[{'match_id': 'J03WPY', 'tracking_frames': 146211, 'owning_team_valid_frames': 146211, 'all_possession_changes': 390, 'active_possession_changes': 324}, {'match_id': 'J03WMX', 'tracking_frames': 145967, 'owning_team_valid_frames': 145967, 'all_possession_changes': 500, 'active_possession_changes': 408}, {'match_id': 'J03WN1', 'tracking_frames': 141561, 'owning_team_valid_frames': 141561, 'all_possession_changes': 467, 'active_possession_changes': 394}, {'match_id': 'J03WOH', 'tracking_frames': 137214, 'owning_team_valid_frames': 137214, 'all_possession_changes': 366, 'active_possession_changes': 301}, {'match_id': 'J03WOY', 'tracking_frames': 142536, 'owning_team_valid_frames': 142536, 'all_possession_changes': 416, 'active_possession_changes': 344}, {'match_id': 'J03WQQ', 'tracking_frames': 142345, 'owning_team_valid_frames': 142345, 'all_possession_changes': 362, 'active_possession_changes': 304}, {'match_id': 'J03WR9', 'tracking_frames': 146810, 'ow

,match_id,tracking_frames,owning_team_valid_frames,all_possession_changes,active_possession_changes
0,J03WPY,146211,146211,390,324
1,J03WMX,145967,145967,500,408
2,J03WN1,141561,141561,467,394
3,J03WOH,137214,137214,366,301
4,J03WOY,142536,142536,416,344
5,J03WQQ,142345,142345,362,304
6,J03WR9,146810,146810,346,278


In [46]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(
    r"C:\VAEP-Track\data\sportec_idsse"
)

MATCH_IDS = [
    "J03WPY",
    "J03WMX",
    "J03WN1",
    "J03WOH",
    "J03WOY",
    "J03WQQ",
    "J03WR9",
]


def analyze_match(match_id):

    tracking_path = DATA_DIR / f"{match_id}_tracking.csv"

    print(f"\n=== {match_id} ===")
    print("File:", tracking_path)

    # --------------------------------------
    # CEK FILE
    # --------------------------------------

    if not tracking_path.exists():
        print("STATUS: FILE TIDAK DITEMUKAN")
        return None

    # --------------------------------------
    # LOAD DATA
    # --------------------------------------

    try:
        tracking = pd.read_csv(tracking_path)
    except Exception as e:
        print("STATUS: GAGAL MEMBACA FILE")
        print("Error:", e)
        return None

    print("Tracking shape:", tracking.shape)

    # --------------------------------------
    # TIMESTAMP
    # --------------------------------------

    tracking["timestamp"] = pd.to_timedelta(
        tracking["timestamp"],
        errors="coerce"
    )

    # --------------------------------------
    # SORT
    # --------------------------------------

    tracking = tracking.sort_values(
        ["period_id", "timestamp", "frame_id"]
    ).reset_index(drop=True)

    # --------------------------------------
    # PREVIOUS OWNER
    # --------------------------------------

    tracking["prev_owning_team"] = (
        tracking
        .groupby("period_id")["ball_owning_team_id"]
        .shift(1)
    )

    # --------------------------------------
    # POSSESSION CHANGE
    # --------------------------------------

    tracking["team_changed"] = (
        tracking["ball_owning_team_id"].notna()
        & tracking["prev_owning_team"].notna()
        & (
            tracking["ball_owning_team_id"]
            != tracking["prev_owning_team"]
        )
    )

    # --------------------------------------
    # SEMUA POSSESSION CHANGE
    # --------------------------------------

    possession_changes = tracking[
        tracking["team_changed"]
    ].copy()

    # --------------------------------------
    # HANYA BOLA AKTIF
    # --------------------------------------

    active_changes = possession_changes[
        possession_changes["ball_state"] == "alive"
    ].copy()

    # --------------------------------------
    # HASIL
    # --------------------------------------

    result = {
        "match_id": match_id,
        "tracking_frames": len(tracking),
        "owning_team_valid_frames": int(
            tracking["ball_owning_team_id"].notna().sum()
        ),
        "all_possession_changes": len(
            possession_changes
        ),
        "active_possession_changes": len(
            active_changes
        ),
    }

    print(
        "Active possession changes:",
        result["active_possession_changes"]
    )

    return result

In [47]:
results = []

for match_id in MATCH_IDS:

    result = analyze_match(match_id)

    if result is not None:
        results.append(result)

print("\n================================")
print("HASIL AKHIR")
print("================================")

print("Jumlah pertandingan berhasil:", len(results))

summary_df = pd.DataFrame(results)

display(summary_df)


=== J03WPY ===
File: C:\VAEP-Track\data\sportec_idsse\J03WPY_tracking.csv
Tracking shape: (146211, 133)
Active possession changes: 324

=== J03WMX ===
File: C:\VAEP-Track\data\sportec_idsse\J03WMX_tracking.csv
Tracking shape: (145967, 137)
Active possession changes: 408

=== J03WN1 ===
File: C:\VAEP-Track\data\sportec_idsse\J03WN1_tracking.csv
Tracking shape: (141561, 125)
Active possession changes: 394

=== J03WOH ===
File: C:\VAEP-Track\data\sportec_idsse\J03WOH_tracking.csv
Tracking shape: (137214, 137)
Active possession changes: 301

=== J03WOY ===
File: C:\VAEP-Track\data\sportec_idsse\J03WOY_tracking.csv
Tracking shape: (142536, 137)
Active possession changes: 344

=== J03WQQ ===
File: C:\VAEP-Track\data\sportec_idsse\J03WQQ_tracking.csv
Tracking shape: (142345, 133)
Active possession changes: 304

=== J03WR9 ===
File: C:\VAEP-Track\data\sportec_idsse\J03WR9_tracking.csv
Tracking shape: (146810, 133)
Active possession changes: 278

HASIL AKHIR
Jumlah pertandingan berhasil: 7


,match_id,tracking_frames,owning_team_valid_frames,all_possession_changes,active_possession_changes
0,J03WPY,146211,146211,390,324
1,J03WMX,145967,145967,500,408
2,J03WN1,141561,141561,467,394
3,J03WOH,137214,137214,366,301
4,J03WOY,142536,142536,416,344
5,J03WQQ,142345,142345,362,304
6,J03WR9,146810,146810,346,278


In [48]:
print(
    "Kolom:",
    summary_df.columns.tolist()
)

print(
    "Total tracking frames:",
    summary_df["tracking_frames"].sum()
)

print(
    "Total possession changes:",
    summary_df["all_possession_changes"].sum()
)

print(
    "Total active possession changes:",
    summary_df["active_possession_changes"].sum()
)

Kolom: ['match_id', 'tracking_frames', 'owning_team_valid_frames', 'all_possession_changes', 'active_possession_changes']
Total tracking frames: 1002644
Total possession changes: 2847
Total active possession changes: 2353


In [49]:
import pandas as pd
from pathlib import Path

# ==========================================
# KONFIGURASI DATA
# ==========================================

DATA_DIR = Path(
    r"C:\VAEP-Track\data\sportec_idsse"
)

MATCH_IDS = [
    "J03WPY",
    "J03WMX",
    "J03WN1",
    "J03WOH",
    "J03WOY",
    "J03WQQ",
    "J03WR9",
]


# ==========================================
# FUNGSI MENGAMBIL ACTIVE POSSESSION CHANGE
# ==========================================

def get_active_possession_changes(match_id):

    tracking_path = DATA_DIR / f"{match_id}_tracking.csv"

    tracking = pd.read_csv(tracking_path)

    # Konversi timestamp
    tracking["timestamp"] = pd.to_timedelta(
        tracking["timestamp"],
        errors="coerce"
    )

    # Urutkan berdasarkan waktu
    tracking = tracking.sort_values(
        ["period_id", "timestamp", "frame_id"]
    ).reset_index(drop=True)

    # Tim yang menguasai bola pada frame sebelumnya
    tracking["prev_owning_team"] = (
        tracking
        .groupby("period_id")["ball_owning_team_id"]
        .shift(1)
    )

    # Perubahan penguasaan
    changes = tracking[
        tracking["ball_owning_team_id"].notna()
        & tracking["prev_owning_team"].notna()
        & (
            tracking["ball_owning_team_id"]
            != tracking["prev_owning_team"]
        )
        & (
            tracking["ball_state"] == "alive"
        )
    ].copy()

    # Tambahkan ID pertandingan
    changes["match_id"] = match_id

    return changes


# ==========================================
# GABUNGKAN 7 PERTANDINGAN
# ==========================================

all_candidates = []

for match_id in MATCH_IDS:

    print(f"Memproses {match_id}...")

    changes = get_active_possession_changes(match_id)

    all_candidates.append(changes)

all_candidates_df = pd.concat(
    all_candidates,
    ignore_index=True
)

print("\n===================================")
print("TOTAL ACTIVE POSSESSION CHANGES")
print("===================================")

print(
    "Jumlah total:",
    len(all_candidates_df)
)

Memproses J03WPY...
Memproses J03WMX...
Memproses J03WN1...
Memproses J03WOH...
Memproses J03WOY...
Memproses J03WQQ...
Memproses J03WR9...

TOTAL ACTIVE POSSESSION CHANGES
Jumlah total: 2353


In [50]:
# ==========================================
# ANALISIS JARAK ANTAR POSSESSION CHANGE
# ==========================================

changes_time = all_candidates_df.copy()

# Urutkan berdasarkan pertandingan, babak, dan waktu
changes_time = changes_time.sort_values(
    ["match_id", "period_id", "timestamp"]
).reset_index(drop=True)

# Timestamp candidate sebelumnya
changes_time["prev_change_timestamp"] = (
    changes_time
    .groupby(["match_id", "period_id"])["timestamp"]
    .shift(1)
)

# Hitung selisih waktu
changes_time["gap_seconds"] = (
    changes_time["timestamp"]
    - changes_time["prev_change_timestamp"]
).dt.total_seconds()

display(
    changes_time[
        [
            "match_id",
            "period_id",
            "timestamp",
            "prev_owning_team",
            "ball_owning_team_id",
            "gap_seconds"
        ]
    ].head(30)
)

,match_id,period_id,timestamp,prev_owning_team,ball_owning_team_id,gap_seconds
0,J03WMX,1,0 days 00:00:23.480000,DFL-CLU-000008,DFL-CLU-00000G,NaN
1,J03WMX,1,0 days 00:00:42.840000,DFL-CLU-00000G,DFL-CLU-000008,19.36
2,J03WMX,1,0 days 00:00:49.520000,DFL-CLU-000008,DFL-CLU-00000G,6.68
3,J03WMX,1,0 days 00:01:22.040000,DFL-CLU-00000G,DFL-CLU-000008,32.52
4,J03WMX,1,0 days 00:01:25.320000,DFL-CLU-000008,DFL-CLU-00000G,3.28
5,J03WMX,1,0 days 00:01:32.760000,DFL-CLU-00000G,DFL-CLU-000008,7.44
6,J03WMX,1,0 days 00:02:09.760000,DFL-CLU-00000G,DFL-CLU-000008,37.00
7,J03WMX,1,0 days 00:02:14.680000,DFL-CLU-000008,DFL-CLU-00000G,4.92
8,J03WMX,1,0 days 00:02:23.560000,DFL-CLU-00000G,DFL-CLU-000008,8.88
9,J03WMX,1,0 days 00:02:25.160000,DFL-CLU-000008,DFL-CLU-00000G,1.60


In [51]:
gap_stats = changes_time["gap_seconds"].dropna()

print("Jumlah gap:", len(gap_stats))

print("\nStatistik gap:")
print(
    gap_stats.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

print("Gap <= 0.5 detik:",
      (gap_stats <= 0.5).sum())

print("Gap <= 1 detik:",
      (gap_stats <= 1.0).sum())

print("Gap <= 2 detik:",
      (gap_stats <= 2.0).sum())

print("Gap > 2 detik:",
      (gap_stats > 2.0).sum())

print("Gap > 5 detik:",
      (gap_stats > 5.0).sum())

Jumlah gap: 2339

Statistik gap:
count    2339.000000
mean       16.854656
std        25.015458
min         0.040000
25%         2.160000
50%         5.640000
75%        23.220000
90%        47.360000
95%        63.900000
max       266.840000
Name: gap_seconds, dtype: float64
Gap <= 0.5 detik: 91
Gap <= 1 detik: 224
Gap <= 2 detik: 529
Gap > 2 detik: 1810
Gap > 5 detik: 1234


ANALISIS REGAIN POSESSION

In [52]:
# ==========================================
# MENCARI POTENTIAL REGAIN ≤ 5 DETIK
# ==========================================

df = all_candidates_df.copy()

# Urutkan berdasarkan match, period, timestamp
df = df.sort_values(
    ["match_id", "period_id", "timestamp"]
).reset_index(drop=True)

# Candidate berikutnya dalam match + period yang sama
df["next_timestamp"] = (
    df.groupby(
        ["match_id", "period_id"]
    )["timestamp"].shift(-1)
)

df["next_losing_team"] = (
    df.groupby(
        ["match_id", "period_id"]
    )["prev_owning_team"].shift(-1)
)

df["next_gaining_team"] = (
    df.groupby(
        ["match_id", "period_id"]
    )["ball_owning_team_id"].shift(-1)
)

# Waktu menuju candidate berikutnya
df["next_gap_seconds"] = (
    df["next_timestamp"] - df["timestamp"]
).dt.total_seconds()

# Apakah possession berikutnya kembali
# ke tim yang kehilangan bola?
df["potential_regain"] = (
    (df["next_gap_seconds"] <= 5.0)
    &
    (
        df["next_gaining_team"]
        == df["prev_owning_team"]
    )
)

print(
    "Total candidate:",
    len(df)
)

print(
    "Potential regain ≤ 5 detik:",
    df["potential_regain"].sum()
)

print(
    "Persentase:",
    round(
        df["potential_regain"].mean() * 100,
        2
    ),
    "%"
)

Total candidate: 2353
Potential regain ≤ 5 detik: 1105
Persentase: 46.96 %


In [53]:
# ==========================================
# CEK DATA POTENTIAL REGAIN
# ==========================================

print("Jumlah event:", len(df))

print("\nKolom:")
print(df.columns.tolist())

print("\nContoh:")
display(df.head())

Jumlah event: 2353

Kolom:
['period_id', 'timestamp', 'frame_id', 'ball_state', 'ball_owning_team_id', 'ball_x', 'ball_y', 'ball_z', 'ball_speed', 'DFL-OBJ-0000NZ_x', 'DFL-OBJ-0000NZ_y', 'DFL-OBJ-0000NZ_d', 'DFL-OBJ-0000NZ_s', 'DFL-OBJ-0000F8_x', 'DFL-OBJ-0000F8_y', 'DFL-OBJ-0000F8_d', 'DFL-OBJ-0000F8_s', 'DFL-OBJ-00028V_x', 'DFL-OBJ-00028V_y', 'DFL-OBJ-00028V_d', 'DFL-OBJ-00028V_s', 'DFL-OBJ-002FXT_x', 'DFL-OBJ-002FXT_y', 'DFL-OBJ-002FXT_d', 'DFL-OBJ-002FXT_s', 'DFL-OBJ-0026RH_x', 'DFL-OBJ-0026RH_y', 'DFL-OBJ-0026RH_d', 'DFL-OBJ-0026RH_s', 'DFL-OBJ-0028FW_x', 'DFL-OBJ-0028FW_y', 'DFL-OBJ-0028FW_d', 'DFL-OBJ-0028FW_s', 'DFL-OBJ-002G5J_x', 'DFL-OBJ-002G5J_y', 'DFL-OBJ-002G5J_d', 'DFL-OBJ-002G5J_s', 'DFL-OBJ-002GMO_x', 'DFL-OBJ-002GMO_y', 'DFL-OBJ-002GMO_d', 'DFL-OBJ-002GMO_s', 'DFL-OBJ-002GM9_x', 'DFL-OBJ-002GM9_y', 'DFL-OBJ-002GM9_d', 'DFL-OBJ-002GM9_s', 'DFL-OBJ-J01H9X_x', 'DFL-OBJ-J01H9X_y', 'DFL-OBJ-J01H9X_d', 'DFL-OBJ-J01H9X_s', 'DFL-OBJ-J01KJ5_x', 'DFL-OBJ-J01KJ5_y', 'DFL-OBJ-J01K

,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,DFL-OBJ-0000NZ_x,...,DFL-OBJ-J01CVM_s,DFL-OBJ-002G0F_x,DFL-OBJ-002G0F_y,DFL-OBJ-002G0F_d,DFL-OBJ-002G0F_s,next_timestamp,next_losing_team,next_gaining_team,next_gap_seconds,potential_regain
0,1,0 days 00:00:23.480000,10587,alive,DFL-CLU-00000G,0.480762,0.319118,0.37,22.85,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:00:42.840000,DFL-CLU-00000G,DFL-CLU-000008,19.36,False
1,1,0 days 00:00:42.840000,11071,alive,DFL-CLU-000008,0.709238,0.309706,0.21,14.81,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:00:49.520000,DFL-CLU-000008,DFL-CLU-00000G,6.68,False
2,1,0 days 00:00:49.520000,11238,alive,DFL-CLU-00000G,0.139524,0.496912,0.54,6.78,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:01:22.040000,DFL-CLU-00000G,DFL-CLU-000008,32.52,False
3,1,0 days 00:01:22.040000,12051,alive,DFL-CLU-000008,0.446857,0.982647,0.09,22.41,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:01:25.320000,DFL-CLU-000008,DFL-CLU-00000G,3.28,True
4,1,0 days 00:01:25.320000,12133,alive,DFL-CLU-00000G,0.341143,0.973529,0.12,24.30,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:01:32.760000,DFL-CLU-00000G,DFL-CLU-000008,7.44,False


In [54]:
# ==========================================
# FILTER POTENTIAL REGAIN
# ==========================================

regain_events = df[
    df["potential_regain"] == True
].copy()

regain_events = regain_events.reset_index(drop=True)

print("Jumlah potential regain:", len(regain_events))

display(
    regain_events.head(10)
)

# ==========================================
# EVENT ID
# ==========================================

regain_events.insert(
    0,
    "event_id",
    range(1, len(regain_events) + 1)
)

display(
    regain_events.head(10)
)

Jumlah potential regain: 1105


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,DFL-OBJ-0000NZ_x,...,DFL-OBJ-J01CVM_s,DFL-OBJ-002G0F_x,DFL-OBJ-002G0F_y,DFL-OBJ-002G0F_d,DFL-OBJ-002G0F_s,next_timestamp,next_losing_team,next_gaining_team,next_gap_seconds,potential_regain
0,1,0 days 00:01:22.040000,12051,alive,DFL-CLU-000008,0.446857,0.982647,0.09,22.41,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:01:25.320000,DFL-CLU-000008,DFL-CLU-00000G,3.28,True
1,1,0 days 00:02:09.760000,13244,alive,DFL-CLU-000008,0.350667,0.941618,0.20,41.53,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:14.680000,DFL-CLU-000008,DFL-CLU-00000G,4.92,True
2,1,0 days 00:02:23.560000,13589,alive,DFL-CLU-000008,0.328000,0.734706,0.26,45.06,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:25.160000,DFL-CLU-000008,DFL-CLU-00000G,1.60,True
3,1,0 days 00:02:25.160000,13629,alive,DFL-CLU-00000G,0.284476,0.773676,0.08,40.43,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:30.040000,DFL-CLU-00000G,DFL-CLU-000008,4.88,True
4,1,0 days 00:04:34.680000,16867,alive,DFL-CLU-000008,0.740190,0.188088,1.97,33.57,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:04:34.920000,DFL-CLU-000008,DFL-CLU-00000G,0.24,True
5,1,0 days 00:04:34.920000,16873,alive,DFL-CLU-00000G,0.758857,0.201029,1.11,35.11,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:04:36.600000,DFL-CLU-00000G,DFL-CLU-000008,1.68,True
6,1,0 days 00:05:04.880000,17622,alive,DFL-CLU-000008,0.839143,0.055294,0.16,15.58,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:06.040000,DFL-CLU-000008,DFL-CLU-00000G,1.16,True
7,1,0 days 00:05:36.680000,18417,alive,DFL-CLU-000008,0.610476,0.719412,0.02,43.66,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:40.040000,DFL-CLU-000008,DFL-CLU-00000G,3.36,True
8,1,0 days 00:05:52.680000,18817,alive,DFL-CLU-000008,0.571905,0.187353,0.11,10.38,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:53.680000,DFL-CLU-000008,DFL-CLU-00000G,1.00,True
9,1,0 days 00:06:01.680000,19042,alive,DFL-CLU-000008,0.705429,0.248529,0.06,25.34,NaN,...,NaN,NaN,NaN,NaN,NaN,0 days 00:06:02.240000,DFL-CLU-000008,DFL-CLU-00000G,0.56,True


,event_id,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,...,DFL-OBJ-J01CVM_s,DFL-OBJ-002G0F_x,DFL-OBJ-002G0F_y,DFL-OBJ-002G0F_d,DFL-OBJ-002G0F_s,next_timestamp,next_losing_team,next_gaining_team,next_gap_seconds,potential_regain
0,1,1,0 days 00:01:22.040000,12051,alive,DFL-CLU-000008,0.446857,0.982647,0.09,22.41,...,NaN,NaN,NaN,NaN,NaN,0 days 00:01:25.320000,DFL-CLU-000008,DFL-CLU-00000G,3.28,True
1,2,1,0 days 00:02:09.760000,13244,alive,DFL-CLU-000008,0.350667,0.941618,0.20,41.53,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:14.680000,DFL-CLU-000008,DFL-CLU-00000G,4.92,True
2,3,1,0 days 00:02:23.560000,13589,alive,DFL-CLU-000008,0.328000,0.734706,0.26,45.06,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:25.160000,DFL-CLU-000008,DFL-CLU-00000G,1.60,True
3,4,1,0 days 00:02:25.160000,13629,alive,DFL-CLU-00000G,0.284476,0.773676,0.08,40.43,...,NaN,NaN,NaN,NaN,NaN,0 days 00:02:30.040000,DFL-CLU-00000G,DFL-CLU-000008,4.88,True
4,5,1,0 days 00:04:34.680000,16867,alive,DFL-CLU-000008,0.740190,0.188088,1.97,33.57,...,NaN,NaN,NaN,NaN,NaN,0 days 00:04:34.920000,DFL-CLU-000008,DFL-CLU-00000G,0.24,True
5,6,1,0 days 00:04:34.920000,16873,alive,DFL-CLU-00000G,0.758857,0.201029,1.11,35.11,...,NaN,NaN,NaN,NaN,NaN,0 days 00:04:36.600000,DFL-CLU-00000G,DFL-CLU-000008,1.68,True
6,7,1,0 days 00:05:04.880000,17622,alive,DFL-CLU-000008,0.839143,0.055294,0.16,15.58,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:06.040000,DFL-CLU-000008,DFL-CLU-00000G,1.16,True
7,8,1,0 days 00:05:36.680000,18417,alive,DFL-CLU-000008,0.610476,0.719412,0.02,43.66,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:40.040000,DFL-CLU-000008,DFL-CLU-00000G,3.36,True
8,9,1,0 days 00:05:52.680000,18817,alive,DFL-CLU-000008,0.571905,0.187353,0.11,10.38,...,NaN,NaN,NaN,NaN,NaN,0 days 00:05:53.680000,DFL-CLU-000008,DFL-CLU-00000G,1.00,True
9,10,1,0 days 00:06:01.680000,19042,alive,DFL-CLU-000008,0.705429,0.248529,0.06,25.34,...,NaN,NaN,NaN,NaN,NaN,0 days 00:06:02.240000,DFL-CLU-000008,DFL-CLU-00000G,0.56,True


In [55]:
# ==========================================
# DISTRIBUSI POTENTIAL REGAIN
# ==========================================

regain_distribution = (
    regain_events
    .groupby("match_id")
    .size()
    .reset_index(name="potential_regain_count")
    .sort_values("match_id")
)

display(regain_distribution)
print(
    "Total:",
    regain_distribution["potential_regain_count"].sum()
)

,match_id,potential_regain_count
0,J03WMX,211
1,J03WN1,204
2,J03WOH,138
3,J03WOY,171
4,J03WPY,148
5,J03WQQ,121
6,J03WR9,112


Total: 1105


CEK DISTRIBUSI BERDASARKAN TIM

In [56]:
# ==========================================
# MEMBENTUK ULANG POTENTIAL REGAIN EVENTS
# ==========================================

# Pastikan df memang memiliki kolom yang dibutuhkan
required_columns = [
    "match_id",
    "period_id",
    "timestamp",
    "frame_id",
    "prev_owning_team",
    "ball_owning_team_id",
    "next_timestamp",
    "next_gap_seconds",
    "next_gaining_team",
    "potential_regain"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Kolom berikut belum tersedia di df: {missing}"
    )


# Ambil hanya baris yang memenuhi potential regain
regain_events = df[
    df["potential_regain"] == True
].copy()

regain_events = regain_events.reset_index(drop=True)


# ==========================================
# BUAT EVENT ID
# ==========================================

regain_events["event_id"] = (
    regain_events.index + 1
)


# ==========================================
# DEFINISI TIM
# ==========================================

# Tim yang kehilangan bola pada T0
regain_events["losing_team"] = (
    regain_events["prev_owning_team"]
)

# Tim yang mendapatkan bola setelah kehilangan
regain_events["opponent_team"] = (
    regain_events["ball_owning_team_id"]
)

# Karena potential_regain == True,
# candidate berikutnya mendapatkan bola kembali
# kepada tim yang sebelumnya kehilangan bola.
regain_events["regain_team"] = (
    regain_events["next_gaining_team"]
)


# ==========================================
# WAKTU REGAIN
# ==========================================

regain_events["regain_timestamp"] = (
    regain_events["next_timestamp"]
)

regain_events["regain_gap_seconds"] = (
    regain_events["next_gap_seconds"]
)


# ==========================================
# CEK HASIL
# ==========================================

print(
    "Jumlah potential regain:",
    len(regain_events)
)

print("\nKolom penting:")
print([
    "event_id",
    "match_id",
    "period_id",
    "timestamp",
    "losing_team",
    "opponent_team",
    "regain_team",
    "regain_timestamp",
    "regain_gap_seconds"
])


display(
    regain_events[
        [
            "event_id",
            "match_id",
            "period_id",
            "timestamp",
            "losing_team",
            "opponent_team",
            "regain_team",
            "regain_timestamp",
            "regain_gap_seconds"
        ]
    ].head(20)
)

Jumlah potential regain: 1105

Kolom penting:
['event_id', 'match_id', 'period_id', 'timestamp', 'losing_team', 'opponent_team', 'regain_team', 'regain_timestamp', 'regain_gap_seconds']


,event_id,match_id,period_id,timestamp,losing_team,opponent_team,regain_team,regain_timestamp,regain_gap_seconds
0,1,J03WMX,1,0 days 00:01:22.040000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:01:25.320000,3.28
1,2,J03WMX,1,0 days 00:02:09.760000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:02:14.680000,4.92
2,3,J03WMX,1,0 days 00:02:23.560000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:02:25.160000,1.60
3,4,J03WMX,1,0 days 00:02:25.160000,DFL-CLU-000008,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:02:30.040000,4.88
4,5,J03WMX,1,0 days 00:04:34.680000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:04:34.920000,0.24
5,6,J03WMX,1,0 days 00:04:34.920000,DFL-CLU-000008,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:04:36.600000,1.68
6,7,J03WMX,1,0 days 00:05:04.880000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:05:06.040000,1.16
7,8,J03WMX,1,0 days 00:05:36.680000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:05:40.040000,3.36
8,9,J03WMX,1,0 days 00:05:52.680000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:05:53.680000,1.00
9,10,J03WMX,1,0 days 00:06:01.680000,DFL-CLU-00000G,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:06:02.240000,0.56


In [57]:
# ==========================================
# DISTRIBUSI REGAIN PER TIM
# ==========================================

team_regain = (
    regain_events
    .groupby("regain_team")
    .size()
    .reset_index(name="regain_count")
    .sort_values(
        "regain_count",
        ascending=False
    )
)

display(team_regain)

print(
    "Total potential regain:",
    team_regain["regain_count"].sum()
)

match_regain = (
    regain_events
    .groupby("match_id")
    .size()
    .reset_index(name="potential_regain_count")
    .sort_values("match_id")
)

display(match_regain)

print(
    "Total:",
    match_regain["potential_regain_count"].sum()
)

print(
    regain_events["regain_gap_seconds"].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

,regain_team,regain_count
6,DFL-CLU-00000P,351
3,DFL-CLU-00000G,106
2,DFL-CLU-00000B,105
1,DFL-CLU-000008,105
8,DFL-CLU-00000S,99
7,DFL-CLU-00000Q,83
4,DFL-CLU-00000H,73
9,DFL-CLU-000011,69
0,DFL-CLU-000005,66
5,DFL-CLU-00000I,48


Total potential regain: 1105


,match_id,potential_regain_count
0,J03WMX,211
1,J03WN1,204
2,J03WOH,138
3,J03WOY,171
4,J03WPY,148
5,J03WQQ,121
6,J03WR9,112


Total: 1105
count    1105.000000
mean        2.251439
std         1.326487
min         0.040000
25%         1.240000
50%         2.080000
75%         3.280000
90%         4.280000
95%         4.640000
max         5.000000
Name: regain_gap_seconds, dtype: float64


In [58]:
# ==========================================
# POTENTIAL REGAIN DALAM 5 DETIK
# VERSI YANG DIKOREKSI
# ==========================================

base = all_candidates_df.copy()

# Urutkan
base = base.sort_values(
    ["match_id", "period_id", "timestamp"]
).reset_index(drop=True)

# Kolom hasil
base["regain_found"] = False
base["regain_timestamp"] = pd.NaT
base["regain_gap_seconds"] = pd.NA
base["regain_from_team"] = pd.NA


# ==========================================
# CARI REGAIN DALAM WINDOW 5 DETIK
# ==========================================

for (match_id, period_id), group in base.groupby(
    ["match_id", "period_id"],
    sort=False
):

    idx = group.index.to_numpy()
    owners = group["ball_owning_team_id"].to_numpy()
    losing_teams = group["prev_owning_team"].to_numpy()

    for i in range(len(group)):

        t0 = group["timestamp"].iloc[i]
        losing_team = losing_teams[i]

        if pd.isna(losing_team):
            continue

        # Batas waktu regain
        end_time = t0 + pd.Timedelta(seconds=5)

        # Cari candidate berikutnya sampai 5 detik
        for j in range(i + 1, len(group)):

            t1 = group["timestamp"].iloc[j]

            if t1 > end_time:
                break

            # Apakah bola kembali ke losing team?
            if owners[j] == losing_team:

                row_idx = idx[i]

                base.loc[
                    row_idx,
                    "regain_found"
                ] = True

                base.loc[
                    row_idx,
                    "regain_timestamp"
                ] = t1

                base.loc[
                    row_idx,
                    "regain_gap_seconds"
                ] = (t1 - t0).total_seconds()

                base.loc[
                    row_idx,
                    "regain_from_team"
                ] = owners[j - 1]

                # ambil regain pertama
                break


# ==========================================
# AMBIL POTENTIAL REGAIN
# ==========================================

regain_events_corrected = base[
    base["regain_found"] == True
].copy()

regain_events_corrected = (
    regain_events_corrected.reset_index(drop=True)
)

regain_events_corrected["event_id"] = (
    regain_events_corrected.index + 1
)


# ==========================================
# INFORMASI TIM
# ==========================================

regain_events_corrected["losing_team"] = (
    regain_events_corrected["prev_owning_team"]
)

regain_events_corrected["opponent_team"] = (
    regain_events_corrected["ball_owning_team_id"]
)


# ==========================================
# HASIL
# ==========================================

print(
    "Total candidate:",
    len(base)
)

print(
    "Potential regain ≤5 detik:",
    len(regain_events_corrected)
)

print(
    "Persentase:",
    round(
        len(regain_events_corrected)
        / len(base)
        * 100,
        2
    ),
    "%"
)


C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:42: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end_time = t0 + pd.Timedelta(seconds=5)
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0 days 00:01:25.320000' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  base.loc[
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:42: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end_time = t0 + pd.Timedelta(seconds=5)
C:\Users\rizal\AppData\Loc

Total candidate: 2353
Potential regain ≤5 detik: 1105
Persentase: 46.96 %


C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:42: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end_time = t0 + pd.Timedelta(seconds=5)
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:42: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end_time = t0 + pd.Timedelta(seconds=5)
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2469902624.py:42: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end_time = t0 + pd.Timedelta(seconds=5)
C:\Users\rizal\AppData\Local\Te

In [59]:
print(
    regain_events_corrected[
        "regain_gap_seconds"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)
print(
    "\nDistribusi per pertandingan:"
)

display(
    regain_events_corrected
    .groupby("match_id")
    .size()
    .reset_index(
        name="potential_regain_count"
    )
)
display(
    regain_events_corrected[
        [
            "event_id",
            "match_id",
            "period_id",
            "timestamp",
            "losing_team",
            "opponent_team",
            "regain_timestamp",
            "regain_from_team",
            "regain_gap_seconds"
        ]
    ].head(20)
)

count     1105.00
unique     125.00
top          0.16
freq        26.00
Name: regain_gap_seconds, dtype: float64

Distribusi per pertandingan:


,match_id,potential_regain_count
0,J03WMX,211
1,J03WN1,204
2,J03WOH,138
3,J03WOY,171
4,J03WPY,148
5,J03WQQ,121
6,J03WR9,112


,event_id,match_id,period_id,timestamp,losing_team,opponent_team,regain_timestamp,regain_from_team,regain_gap_seconds
0,1,J03WMX,1,0 days 00:01:22.040000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:01:25.320000,DFL-CLU-000008,3.28
1,2,J03WMX,1,0 days 00:02:09.760000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:02:14.680000,DFL-CLU-000008,4.92
2,3,J03WMX,1,0 days 00:02:23.560000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:02:25.160000,DFL-CLU-000008,1.6
3,4,J03WMX,1,0 days 00:02:25.160000,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:02:30.040000,DFL-CLU-00000G,4.88
4,5,J03WMX,1,0 days 00:04:34.680000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:04:34.920000,DFL-CLU-000008,0.24
5,6,J03WMX,1,0 days 00:04:34.920000,DFL-CLU-000008,DFL-CLU-00000G,0 days 00:04:36.600000,DFL-CLU-00000G,1.68
6,7,J03WMX,1,0 days 00:05:04.880000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:05:06.040000,DFL-CLU-000008,1.16
7,8,J03WMX,1,0 days 00:05:36.680000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:05:40.040000,DFL-CLU-000008,3.36
8,9,J03WMX,1,0 days 00:05:52.680000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:05:53.680000,DFL-CLU-000008,1.0
9,10,J03WMX,1,0 days 00:06:01.680000,DFL-CLU-00000G,DFL-CLU-000008,0 days 00:06:02.240000,DFL-CLU-000008,0.56


In [60]:
# ==========================================
# AMBIL 1 EPISODE POTENTIAL REGAIN
# ==========================================

episode = regain_events_corrected[
    regain_events_corrected["event_id"] == 1
].iloc[0]

print("=== EPISODE 1 ===")
print("Match        :", episode["match_id"])
print("Period       :", episode["period_id"])
print("Loss time    :", episode["timestamp"])
print("Losing team  :", episode["losing_team"])
print("Opponent     :", episode["opponent_team"])
print("Regain time  :", episode["regain_timestamp"])
print("Regain gap   :", episode["regain_gap_seconds"], "detik")

# ==========================================
# LOAD TRACKING EPISODE
# ==========================================

match_id = episode["match_id"]

tracking_path = DATA_DIR / f"{match_id}_tracking.csv"

episode_tracking = pd.read_csv(tracking_path)

episode_tracking["timestamp"] = pd.to_timedelta(
    episode_tracking["timestamp"],
    errors="coerce"
)

episode_tracking = episode_tracking.sort_values(
    ["period_id", "timestamp", "frame_id"]
).reset_index(drop=True)

print("Tracking shape:", episode_tracking.shape)

# ==========================================
# EPISODE TRACKING WINDOW
# ==========================================

t0 = episode["timestamp"]
t1 = episode["regain_timestamp"]
period_id = episode["period_id"]

window_start = t0 - pd.Timedelta(seconds=2)
window_end = t1 + pd.Timedelta(seconds=1)

episode_window = episode_tracking[
    (episode_tracking["period_id"] == period_id)
    & (episode_tracking["timestamp"] >= window_start)
    & (episode_tracking["timestamp"] <= window_end)
].copy()

print("=== EPISODE WINDOW ===")
print("Mulai :", window_start)
print("Loss  :", t0)
print("Regain :", t1)
print("Selesai:", window_end)
print("Jumlah frame:", len(episode_window))

display(
    episode_window[
        [
            "timestamp",
            "frame_id",
            "ball_state",
            "ball_owning_team_id",
            "ball_x",
            "ball_y"
        ]
    ].iloc[
        max(0, len(episode_window)//2 - 10):
        min(len(episode_window), len(episode_window)//2 + 10)
    ]
)

# ==========================================
# SEKITAR LOSS
# ==========================================

loss_context = episode_window[
    (episode_window["timestamp"] >= t0 - pd.Timedelta(seconds=0.5))
    & (episode_window["timestamp"] <= t0 + pd.Timedelta(seconds=0.5))
]

display(
    loss_context[
        [
            "timestamp",
            "frame_id",
            "ball_state",
            "ball_owning_team_id",
            "ball_x",
            "ball_y"
        ]
    ]
)

# ==========================================
# SEKITAR REGAIN
# ==========================================

regain_context = episode_window[
    (episode_window["timestamp"] >= t1 - pd.Timedelta(seconds=0.5))
    & (episode_window["timestamp"] <= t1 + pd.Timedelta(seconds=0.5))
]

display(
    regain_context[
        [
            "timestamp",
            "frame_id",
            "ball_state",
            "ball_owning_team_id",
            "ball_x",
            "ball_y"
        ]
    ]
)

# ==========================================
# FRAME TERDEKAT DENGAN LOSS
# ==========================================

loss_row = episode_tracking[
    episode_tracking["period_id"] == period_id
].copy()

loss_row["time_diff"] = (
    loss_row["timestamp"] - t0
).abs()

loss_row = loss_row.sort_values("time_diff").iloc[0]

print("Loss target :", t0)
print("Frame loss  :", loss_row["frame_id"])
print("Timestamp   :", loss_row["timestamp"])
print("Selisih     :", loss_row["time_diff"])

# ==========================================
# FRAME TERDEKAT DENGAN REGAIN
# ==========================================

regain_row = episode_tracking[
    episode_tracking["period_id"] == period_id
].copy()

regain_row["time_diff"] = (
    regain_row["timestamp"] - t1
).abs()

regain_row = regain_row.sort_values(
    "time_diff"
).iloc[0]

print("Regain target:", t1)
print("Frame regain :", regain_row["frame_id"])
print("Timestamp    :", regain_row["timestamp"])
print("Selisih      :", regain_row["time_diff"])

=== EPISODE 1 ===
Match        : J03WMX
Period       : 1
Loss time    : 0 days 00:01:22.040000
Losing team  : DFL-CLU-00000G
Opponent     : DFL-CLU-000008
Regain time  : 0 days 00:01:25.320000
Regain gap   : 3.2800000000000002 detik
Tracking shape: (145967, 137)
=== EPISODE WINDOW ===
Mulai : 0 days 00:01:20.040000
Loss  : 0 days 00:01:22.040000
Regain : 0 days 00:01:25.320000
Selesai: 0 days 00:01:26.320000
Jumlah frame: 158


C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:47: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  window_start = t0 - pd.Timedelta(seconds=2)
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:48: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  window_end = t1 + pd.Timedelta(seconds=1)


,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y
2070,0 days 00:01:22.800000,12070,alive,DFL-CLU-000008,0.495905,0.982941
2071,0 days 00:01:22.840000,12071,alive,DFL-CLU-000008,0.497714,0.982647
2072,0 days 00:01:22.880000,12072,alive,DFL-CLU-000008,0.499619,0.982647
2073,0 days 00:01:22.920000,12073,alive,DFL-CLU-000008,0.501524,0.982353
2074,0 days 00:01:22.960000,12074,alive,DFL-CLU-000008,0.503429,0.982059
2075,0 days 00:01:23,12075,alive,DFL-CLU-000008,0.505143,0.981324
2076,0 days 00:01:23.040000,12076,alive,DFL-CLU-000008,0.506952,0.981324
2077,0 days 00:01:23.080000,12077,alive,DFL-CLU-000008,0.508762,0.981029
2078,0 days 00:01:23.120000,12078,alive,DFL-CLU-000008,0.510667,0.981029
2079,0 days 00:01:23.160000,12079,alive,DFL-CLU-000008,0.512381,0.980441


C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:84: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  (episode_window["timestamp"] >= t0 - pd.Timedelta(seconds=0.5))
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:85: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (episode_window["timestamp"] <= t0 + pd.Timedelta(seconds=0.5))


,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y
2039,0 days 00:01:21.560000,12039,alive,DFL-CLU-00000G,0.437143,0.991176
2040,0 days 00:01:21.600000,12040,alive,DFL-CLU-00000G,0.437810,0.991176
2041,0 days 00:01:21.640000,12041,alive,DFL-CLU-00000G,0.437619,0.991176
2042,0 days 00:01:21.680000,12042,alive,DFL-CLU-00000G,0.438381,0.991176
2043,0 days 00:01:21.720000,12043,alive,DFL-CLU-00000G,0.438190,0.991176
2044,0 days 00:01:21.760000,12044,alive,DFL-CLU-00000G,0.438381,0.991176
2045,0 days 00:01:21.800000,12045,alive,DFL-CLU-00000G,0.438571,0.991176
2046,0 days 00:01:21.840000,12046,alive,DFL-CLU-00000G,0.438857,0.991176
2047,0 days 00:01:21.880000,12047,alive,DFL-CLU-00000G,0.439429,0.991176
2048,0 days 00:01:21.920000,12048,alive,DFL-CLU-00000G,0.440000,0.991324


C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:106: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  (episode_window["timestamp"] >= t1 - pd.Timedelta(seconds=0.5))
C:\Users\rizal\AppData\Local\Temp\ipykernel_2544\2383567007.py:107: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (episode_window["timestamp"] <= t1 + pd.Timedelta(seconds=0.5))


,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y
2121,0 days 00:01:24.840000,12121,alive,DFL-CLU-000008,0.373810,0.979265
2122,0 days 00:01:24.880000,12122,alive,DFL-CLU-000008,0.370571,0.978971
2123,0 days 00:01:24.920000,12123,alive,DFL-CLU-000008,0.367429,0.978824
2124,0 days 00:01:24.960000,12124,alive,DFL-CLU-000008,0.364095,0.978382
2125,0 days 00:01:25,12125,alive,DFL-CLU-000008,0.360952,0.977647
2126,0 days 00:01:25.040000,12126,alive,DFL-CLU-000008,0.357905,0.977206
2127,0 days 00:01:25.080000,12127,alive,DFL-CLU-000008,0.355143,0.976765
2128,0 days 00:01:25.120000,12128,alive,DFL-CLU-000008,0.352762,0.976324
2129,0 days 00:01:25.160000,12129,alive,DFL-CLU-000008,0.350286,0.975882
2130,0 days 00:01:25.200000,12130,alive,DFL-CLU-000008,0.348000,0.975441


Loss target : 0 days 00:01:22.040000
Frame loss  : 12051
Timestamp   : 0 days 00:01:22.040000
Selisih     : 0 days 00:00:00
Regain target: 0 days 00:01:25.320000
Frame regain : 12133
Timestamp    : 0 days 00:01:25.320000
Selisih      : 0 days 00:00:00


LANGKAH KE 2

In [64]:
# ==========================================
# LOAD EVENT KHUSUS MATCH EPISODE
# ==========================================
print("Match:", episode["match_id"])
print("Period:", episode["period_id"])
print("Loss:", episode["timestamp"])
print("Regain:", episode["regain_timestamp"])


match_id = episode["match_id"]

events_path = DATA_DIR / f"{match_id}_events.csv"

events_episode = pd.read_csv(events_path)

events_episode["timestamp"] = pd.to_timedelta(
    events_episode["timestamp"],
    errors="coerce"
)

print("Event shape:", events_episode.shape)
print("Match:", match_id)

# ==========================================
# PLAYER → TEAM MAPPING
# ==========================================

player_team_episode = (
    events_episode[
        ["player_id", "team_id"]
    ]
    .dropna()
    .drop_duplicates()
)

print(
    "Jumlah player-team mapping:",
    len(player_team_episode)
)

display(
    player_team_episode.head(20)
)

# ==========================================
# IDENTIFIKASI PLAYER DARI KOLOM TRACKING
# ==========================================

player_ids_tracking = sorted({
    col[:-2]
    for col in episode_tracking.columns
    if (
        col.startswith("DFL-OBJ-")
        and col.endswith("_x")
    )
})

print(
    "Jumlah player/object dari tracking:",
    len(player_ids_tracking)
)

for player_id in player_ids_tracking:
    print(player_id)
    
# ==========================================
# PEMAIN PADA SAAT LOSS
# ==========================================

loss_nodes = []

for player_id in player_ids_tracking:

    x_col = f"{player_id}_x"
    y_col = f"{player_id}_y"
    s_col = f"{player_id}_s"

    # Pastikan kolom tersedia
    if (
        x_col not in episode_tracking.columns
        or y_col not in episode_tracking.columns
    ):
        continue

    x = frame_loss[x_col]
    y = frame_loss[y_col]

    # Hanya pemain dengan posisi valid
    if pd.isna(x) or pd.isna(y):
        continue

    # Cari team pemain
    team_rows = player_team_episode[
        player_team_episode["player_id"] == player_id
    ]

    if len(team_rows) == 0:
        continue

    team_id = team_rows["team_id"].iloc[0]

    speed = frame_loss[s_col]

    loss_nodes.append({
        "player_id": player_id,
        "team_id": team_id,
        "x": x,
        "y": y,
        "speed": speed
    })

loss_nodes_df = pd.DataFrame(loss_nodes)

print(
    "Jumlah pemain aktif saat LOSS:",
    len(loss_nodes_df)
)

display(loss_nodes_df)

ball_x = frame_loss["ball_x"]
ball_y = frame_loss["ball_y"]

print("Ball X:", ball_x)
print("Ball Y:", ball_y)

# ==========================================
# DISTANCE PLAYER → BALL
# ==========================================

loss_nodes_df["distance_to_ball"] = (
    (
        (loss_nodes_df["x"] - ball_x) ** 2
        +
        (loss_nodes_df["y"] - ball_y) ** 2
    ) ** 0.5
)

loss_nodes_df = loss_nodes_df.sort_values(
    "distance_to_ball"
).reset_index(drop=True)

display(loss_nodes_df)

losing_team = episode["losing_team"]

losing_players = loss_nodes_df[
    loss_nodes_df["team_id"] == losing_team
].copy()

losing_players = losing_players.sort_values(
    "distance_to_ball"
)

print(
    "Jumlah pemain tim yang kehilangan bola:",
    len(losing_players)
)

display(losing_players)

Match: J03WMX
Period: 1
Loss: 0 days 00:01:22.040000
Regain: 0 days 00:01:25.320000
Event shape: (1789, 19)
Match: J03WMX
Jumlah player-team mapping: 32


,player_id,team_id
0,DFL-OBJ-0027G6,DFL-CLU-00000G
1,DFL-OBJ-0027KL,DFL-CLU-00000G
3,DFL-OBJ-0002BO,DFL-CLU-000008
6,DFL-OBJ-J01B8N,DFL-CLU-00000G
8,DFL-OBJ-0002AU,DFL-CLU-00000G
9,DFL-OBJ-0027G0,DFL-CLU-00000G
11,DFL-OBJ-J017RE,DFL-CLU-00000G
13,DFL-OBJ-0026PM,DFL-CLU-00000G
14,DFL-OBJ-0027B6,DFL-CLU-000008
15,DFL-OBJ-0002DR,DFL-CLU-00000G


Jumlah player/object dari tracking: 32
DFL-OBJ-0000IA
DFL-OBJ-0000RP
DFL-OBJ-00012X
DFL-OBJ-000191
DFL-OBJ-000270
DFL-OBJ-0002AU
DFL-OBJ-0002BO
DFL-OBJ-0002DR
DFL-OBJ-0002F5
DFL-OBJ-0002HE
DFL-OBJ-0026PM
DFL-OBJ-0026ZI
DFL-OBJ-0027AX
DFL-OBJ-0027B6
DFL-OBJ-0027G0
DFL-OBJ-0027G6
DFL-OBJ-0027KL
DFL-OBJ-0027V2
DFL-OBJ-0028BD
DFL-OBJ-002G4A
DFL-OBJ-002G89
DFL-OBJ-002GBW
DFL-OBJ-002GCR
DFL-OBJ-002GKO
DFL-OBJ-J015S4
DFL-OBJ-J017RE
DFL-OBJ-J01APO
DFL-OBJ-J01B8N
DFL-OBJ-J01BGM
DFL-OBJ-J01D1W
DFL-OBJ-J01DVC
DFL-OBJ-J01N65
Jumlah pemain aktif saat LOSS: 22


,player_id,team_id,x,y,speed
0,DFL-OBJ-0000IA,DFL-CLU-00000G,0.542095,0.883382,4.87
1,DFL-OBJ-00012X,DFL-CLU-000008,0.445048,0.976471,8.45
2,DFL-OBJ-000270,DFL-CLU-000008,0.444857,0.639265,5.08
3,DFL-OBJ-0002AU,DFL-CLU-00000G,0.443048,0.992941,8.73
4,DFL-OBJ-0002BO,DFL-CLU-000008,0.598286,0.266324,6.41
5,DFL-OBJ-0002DR,DFL-CLU-00000G,0.137905,0.580735,2.81
6,DFL-OBJ-0002F5,DFL-CLU-00000G,0.324000,0.633382,5.78
7,DFL-OBJ-0002HE,DFL-CLU-000008,0.854286,0.570441,4.44
8,DFL-OBJ-0026PM,DFL-CLU-00000G,0.610762,0.082794,3.92
9,DFL-OBJ-0027AX,DFL-CLU-000008,0.360952,0.727647,10.49


Ball X: 0.4468571428571428
Ball Y: 0.9826470588235292


,player_id,team_id,x,y,speed,distance_to_ball
0,DFL-OBJ-00012X,DFL-CLU-000008,0.445048,0.976471,8.45,0.006436
1,DFL-OBJ-0002AU,DFL-CLU-00000G,0.443048,0.992941,8.73,0.010976
2,DFL-OBJ-0028BD,DFL-CLU-000008,0.402095,0.938971,12.54,0.062540
3,DFL-OBJ-0000IA,DFL-CLU-00000G,0.542095,0.883382,4.87,0.137564
4,DFL-OBJ-002G4A,DFL-CLU-000008,0.463905,0.841912,8.92,0.141764
5,DFL-OBJ-0027G0,DFL-CLU-00000G,0.416571,0.837206,7.75,0.148561
6,DFL-OBJ-J01D1W,DFL-CLU-000008,0.564952,0.891912,15.60,0.148927
7,DFL-OBJ-0027KL,DFL-CLU-00000G,0.286952,0.910294,9.15,0.175512
8,DFL-OBJ-0027AX,DFL-CLU-000008,0.360952,0.727647,10.49,0.269081
9,DFL-OBJ-002GKO,DFL-CLU-000008,0.529714,0.650294,8.36,0.342526


Jumlah pemain tim yang kehilangan bola: 11


,player_id,team_id,x,y,speed,distance_to_ball
1,DFL-OBJ-0002AU,DFL-CLU-00000G,0.443048,0.992941,8.73,0.010976
3,DFL-OBJ-0000IA,DFL-CLU-00000G,0.542095,0.883382,4.87,0.137564
5,DFL-OBJ-0027G0,DFL-CLU-00000G,0.416571,0.837206,7.75,0.148561
7,DFL-OBJ-0027KL,DFL-CLU-00000G,0.286952,0.910294,9.15,0.175512
11,DFL-OBJ-J01B8N,DFL-CLU-00000G,0.454667,0.622941,4.06,0.359791
12,DFL-OBJ-0002F5,DFL-CLU-00000G,0.324000,0.633382,5.78,0.370243
14,DFL-OBJ-0002DR,DFL-CLU-00000G,0.137905,0.580735,2.81,0.506937
16,DFL-OBJ-J017RE,DFL-CLU-00000G,0.282476,0.443529,5.42,0.563621
17,DFL-OBJ-0027G6,DFL-CLU-00000G,0.517048,0.417059,0.28,0.569927
20,DFL-OBJ-J01APO,DFL-CLU-00000G,0.377143,0.198088,8.87,0.787650


In [65]:
display(
    losing_players[
        [
            "player_id",
            "team_id",
            "x",
            "y",
            "speed",
            "distance_to_ball"
        ]
    ]
)

,player_id,team_id,x,y,speed,distance_to_ball
1,DFL-OBJ-0002AU,DFL-CLU-00000G,0.443048,0.992941,8.73,0.010976
3,DFL-OBJ-0000IA,DFL-CLU-00000G,0.542095,0.883382,4.87,0.137564
5,DFL-OBJ-0027G0,DFL-CLU-00000G,0.416571,0.837206,7.75,0.148561
7,DFL-OBJ-0027KL,DFL-CLU-00000G,0.286952,0.910294,9.15,0.175512
11,DFL-OBJ-J01B8N,DFL-CLU-00000G,0.454667,0.622941,4.06,0.359791
12,DFL-OBJ-0002F5,DFL-CLU-00000G,0.324000,0.633382,5.78,0.370243
14,DFL-OBJ-0002DR,DFL-CLU-00000G,0.137905,0.580735,2.81,0.506937
16,DFL-OBJ-J017RE,DFL-CLU-00000G,0.282476,0.443529,5.42,0.563621
17,DFL-OBJ-0027G6,DFL-CLU-00000G,0.517048,0.417059,0.28,0.569927
20,DFL-OBJ-J01APO,DFL-CLU-00000G,0.377143,0.198088,8.87,0.787650


In [66]:
# ==========================================
# TIME SERIES JARAK PEMAIN KE BOLA
# SELAMA LOSS -> REGAIN
# ==========================================

losing_team = episode["losing_team"]

# Ambil semua player ID yang terpetakan ke team
losing_player_ids = (
    player_team_episode[
        player_team_episode["team_id"] == losing_team
    ]["player_id"]
    .dropna()
    .unique()
    .tolist()
)

print("Jumlah pemain losing team:", len(losing_player_ids))

distance_records = []

# Tracking dari loss sampai regain
episode_frames = episode_tracking[
    (episode_tracking["period_id"] == period_id)
    & (episode_tracking["timestamp"] >= t0)
    & (episode_tracking["timestamp"] <= t1)
].copy()

for _, frame in episode_frames.iterrows():

    timestamp = frame["timestamp"]

    ball_x = frame["ball_x"]
    ball_y = frame["ball_y"]

    relative_time = (
        timestamp - t0
    ).total_seconds()

    for player_id in losing_player_ids:

        x_col = f"{player_id}_x"
        y_col = f"{player_id}_y"
        s_col = f"{player_id}_s"

        if (
            x_col not in episode_tracking.columns
            or y_col not in episode_tracking.columns
        ):
            continue

        x = frame[x_col]
        y = frame[y_col]

        if pd.isna(x) or pd.isna(y):
            continue

        distance = (
            (
                (x - ball_x) ** 2
                +
                (y - ball_y) ** 2
            ) ** 0.5
        )

        speed = frame[s_col]

        distance_records.append({
            "timestamp": timestamp,
            "relative_time": relative_time,
            "frame_id": frame["frame_id"],
            "player_id": player_id,
            "distance_to_ball": distance,
            "speed": speed
        })

distance_ts = pd.DataFrame(distance_records)

print(
    "Jumlah record:",
    len(distance_ts)
)

display(distance_ts.head(30))

# ==========================================
# PERUBAHAN JARAK TERHADAP SAAT LOSS
# ==========================================

initial_distance = (
    distance_ts
    .groupby("player_id")["distance_to_ball"]
    .transform("first")
)

distance_ts["distance_change"] = (
    distance_ts["distance_to_ball"]
    - initial_distance
)

display(
    distance_ts.head(30)
)

# ==========================================
# PERUBAHAN JARAK DARI LOSS -> REGAIN
# ==========================================

player_response = (
    distance_ts
    .sort_values(
        ["player_id", "timestamp"]
    )
    .groupby("player_id")
    .agg(
        initial_distance=(
            "distance_to_ball",
            "first"
        ),
        final_distance=(
            "distance_to_ball",
            "last"
        ),
        min_distance=(
            "distance_to_ball",
            "min"
        ),
        initial_speed=(
            "speed",
            "first"
        ),
        max_speed=(
            "speed",
            "max"
        )
    )
    .reset_index()
)

player_response["distance_change"] = (
    player_response["final_distance"]
    - player_response["initial_distance"]
)

player_response["distance_reduction"] = (
    player_response["initial_distance"]
    - player_response["min_distance"]
)

player_response = player_response.sort_values(
    "distance_reduction",
    ascending=False
)

display(player_response)

# ==========================================
# FITUR KOLEKTIF LOSING TEAM
# ==========================================

team_pressure_ts = (
    distance_ts
    .groupby(
        ["timestamp", "relative_time", "frame_id"]
    )
    .agg(
        nearest_player_distance=(
            "distance_to_ball",
            "min"
        ),
        mean_player_distance=(
            "distance_to_ball",
            "mean"
        ),
        players_count=(
            "player_id",
            "nunique"
        )
    )
    .reset_index()
)

display(team_pressure_ts.head(30))

Jumlah pemain losing team: 16
Jumlah record: 913


,timestamp,relative_time,frame_id,player_id,distance_to_ball,speed
0,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027G6,0.569927,0.28
1,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027KL,0.175512,9.15
2,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J01B8N,0.359791,4.06
3,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002AU,0.010976,8.73
4,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027G0,0.148561,7.75
5,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J017RE,0.563621,5.42
6,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0026PM,0.914658,3.92
7,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002DR,0.506937,2.81
8,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002F5,0.370243,5.78
9,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J01APO,0.787650,8.87


,timestamp,relative_time,frame_id,player_id,distance_to_ball,speed,distance_change
0,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027G6,0.569927,0.28,0.000000
1,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027KL,0.175512,9.15,0.000000
2,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J01B8N,0.359791,4.06,0.000000
3,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002AU,0.010976,8.73,0.000000
4,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0027G0,0.148561,7.75,0.000000
5,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J017RE,0.563621,5.42,0.000000
6,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0026PM,0.914658,3.92,0.000000
7,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002DR,0.506937,2.81,0.000000
8,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-0002F5,0.370243,5.78,0.000000
9,0 days 00:01:22.040000,0.00,12051,DFL-OBJ-J01APO,0.787650,8.87,0.000000


,player_id,initial_distance,final_distance,min_distance,initial_speed,max_speed,distance_change,distance_reduction
7,DFL-OBJ-0027KL,0.175512,0.008537,0.008537,9.15,12.99,-0.166975,0.166975
9,DFL-OBJ-J01APO,0.787650,0.662204,0.661956,8.87,9.79,-0.125447,0.125694
3,DFL-OBJ-0002F5,0.370243,0.282505,0.282083,5.78,10.80,-0.087737,0.088160
2,DFL-OBJ-0002DR,0.506937,0.426528,0.426528,2.81,12.12,-0.080408,0.080408
8,DFL-OBJ-J017RE,0.563621,0.496359,0.495829,5.42,7.65,-0.067262,0.067792
5,DFL-OBJ-0027G0,0.148561,0.110148,0.089714,7.75,7.75,-0.038413,0.058847
4,DFL-OBJ-0026PM,0.914658,0.870179,0.864050,3.92,10.44,-0.044480,0.050609
0,DFL-OBJ-0000IA,0.137564,0.273135,0.100110,4.87,13.41,0.135571,0.037454
6,DFL-OBJ-0027G6,0.569927,0.553061,0.537253,0.28,13.67,-0.016866,0.032674
10,DFL-OBJ-J01B8N,0.359791,0.367204,0.347054,4.06,5.47,0.007413,0.012736


,timestamp,relative_time,frame_id,nearest_player_distance,mean_player_distance,players_count
0,0 days 00:01:22.040000,0.00,12051,0.010976,0.413222,11
1,0 days 00:01:22.080000,0.04,12052,0.012425,0.412560,11
2,0 days 00:01:22.120000,0.08,12053,0.011798,0.415477,11
3,0 days 00:01:22.160000,0.12,12054,0.013070,0.415453,11
4,0 days 00:01:22.200000,0.16,12055,0.014554,0.415348,11
5,0 days 00:01:22.240000,0.20,12056,0.016068,0.415118,11
6,0 days 00:01:22.280000,0.24,12057,0.017274,0.415058,11
7,0 days 00:01:22.320000,0.28,12058,0.018645,0.414939,11
8,0 days 00:01:22.360000,0.32,12059,0.020003,0.414487,11
9,0 days 00:01:22.400000,0.36,12060,0.020758,0.414669,11
